# QuantiPhy · Code-as-World-VL-4B · Full validation

Chạy độc lập toàn bộ 158 câu / 24 video validation trên **GPU T4 ×2** (đã tích hợp sẵn cấu hình `PROFILE = "replica_fp16"`, `PARTITION_COUNT = 1` từ thực nghiệm trên 2×T4, không cần gắn archive sanity).

Chọn **Save & Run All** để lưu `quantiphy_full.tar` và `quantiphy_summary.json`. Nếu cần chạy tiếp (resume), gắn archive full cũ qua **Add Input** và đặt `RESUME_ARCHIVE = "auto"` (hoặc đường dẫn file). Giữ nguyên `SOURCE_REF`, `PROFILE`, `PARTITION_COUNT` và `PARTITION_INDEX` khi resume.

Chọn **GPU T4 ×2** và bật **Internet** trên Kaggle. Notebook lấy mã từ `leetranwyng/quantiphy_trannl`. Khi repo còn Private, thêm secret `GITHUB_TOKEN` trên Kaggle bằng token GitHub có quyền đọc repo. Khi repo Public, notebook tự clone bằng HTTPS.

Sau khi chạy, giải nén archive trong tab **Output** và mở `run/index.html` để xem video, frame, prompt, token, tensor vision và trace từng câu. `python run/verify.py` kiểm tra output offline.

## Cấu hình

In [ ]:
REPO_URL = "https://github.com/leetranwyng/quantiphy_trannl.git"
SOURCE_REF = "main"
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # Tên secret trên Kaggle dùng khi repo còn Private.

# Để trống để tự tải snapshot đã ghim. Hoặc điền thư mục snapshot từ /kaggle/input.
MODEL_DIR = ""
DATASET_DIR = ""
WHEELHOUSE = ""

# Cấu hình đã chuẩn hóa từ kết quả đo thực tế trên Kaggle 2×T4 (~9.99 GB VRAM/GPU, ~4 phút cho 158 câu).
PROFILE = "replica_fp16"  # "replica_fp16" (2 worker song song trên 2×T4) hoặc "split_fp16".
OUTPUT_BUDGET_GIB = 15.0
SANITY_ARCHIVE = ""  # Không bắt buộc; để trống khi chạy độc lập.
RESUME_ARCHIVE = ""  # Để trống, đặt "auto" hoặc đường dẫn quantiphy_full.tar cũ khi cần resume.
PARTITION_COUNT = 1
PARTITION_INDEX = 0  # Đánh số từ 0.
MERGE_ARCHIVES = []  # Điền các archive hoàn tất để hợp nhất trên CPU nếu chia nhiều partition.
WORK_MINUTES = 480

## Chuẩn bị repo và thư viện

In [ ]:
import base64, os, subprocess, sys, tempfile
from pathlib import Path

source = Path(tempfile.mkdtemp(prefix="quantiphy-")) / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
clone = ["git", "-c", "credential.helper=", "clone", "--quiet", REPO_URL, str(source)]
if subprocess.run(clone, env=env).returncode:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    subprocess.run(clone, check=True, env=dict(env, GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraHeader",
        GIT_CONFIG_VALUE_0="Authorization: Basic " + auth))
    del token, auth
remote_ref = f"refs/remotes/origin/{SOURCE_REF}"
branch = subprocess.run(["git", "show-ref", "--verify", "--quiet", remote_ref], cwd=source).returncode == 0
subprocess.run(["git", "checkout", "--quiet", "--detach", remote_ref if branch else SOURCE_REF], cwd=source, check=True)
print("Source commit", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source, text=True).strip())
install = [sys.executable, "-m", "pip", "install", "-r", str(source / "code_as_world/kaggle/requirements.txt")]
subprocess.run(install + (["--no-index", "--find-links", WHEELHOUSE] if WHEELHOUSE else []), check=True)


## Chạy và lưu output

In [ ]:
# Execute the launcher from this checkout, even after rerunning earlier cells.
import runpy
sys.path.insert(0, str(source))
launch = runpy.run_path(str(source / "code_as_world/kaggle/notebook.py"))["launch"]
SUMMARY = launch(source, "full", globals())
